In [6]:
import pandas as pd
import json
from IPython.display import display

file_name = "2026-03-20_11-46-40"
with open(f"../data/raw/variant_16/{file_name}.json", "r", encoding="utf-8") as f:
    data = json.load(f)

features = data["features"]
df = pd.json_normalize(features)

records = []
for feature in data["features"]:
    props = feature["properties"]
    coords = feature["geometry"]["coordinates"]

    record = {
        "id" : str(feature["id"]),
        "time": pd.to_datetime(props["time"], unit="ms", errors="coerce"),
        "mag": float(props["mag"]),
        "place": str(props["place"]),
        "latitude": float(coords[1]),
        "longitude": float(coords[0]),
        "depth_km": float(coords[2]),
        "alert": str(props["alert"]),
        "tsunami": bool(props["tsunami"]),
        "sig": int(props["sig"]),
    }
    records.append(record)

df = pd.DataFrame(records)

display(df.head())
print("\n", df.dtypes)
print("\n", df.shape)
print("\n", df.columns)
print("/n", "Количество дубликатов по time:", df.duplicated(subset=["time"]).sum())

clean_df = df.drop_duplicates(subset=["time"], keep="first")
clean_df = clean_df.sort_values("time").reset_index(drop=True)

clean_df.to_csv(f"../data/normalized/variant_16/{file_name}.csv")
print("\n", "normalized данные сохранены")

display(clean_df.head(50))


,id,time,mag,place,latitude,longitude,depth_km,alert,tsunami,sig
0,us6000sh9j,2026-03-18 21:23:02.008,4.7,"52 km NE of Misawa, Japan",40.9524,141.9080,64.376,None,False,340
1,us6000sh54,2026-03-18 10:08:43.719,4.8,"Izu Islands, Japan region",32.4671,141.7638,10.000,None,False,355
2,us6000sh22,2026-03-17 23:39:22.117,4.3,"Izu Islands, Japan region",30.3452,137.4131,503.225,None,False,284
3,us6000sh1g,2026-03-17 21:13:32.230,4.5,"80 km S of Kushiro, Japan",42.2522,144.3656,35.000,None,False,312
4,us6000sgkn,2026-03-16 11:37:49.386,4.6,"37 km SE of Itō, Japan",34.7732,139.4133,121.973,None,False,326



 id                      str
time         datetime64[ms]
mag                 float64
place                   str
latitude            float64
longitude           float64
depth_km            float64
alert                   str
tsunami                bool
sig                   int64
dtype: object

 (47, 10)

 Index(['id', 'time', 'mag', 'place', 'latitude', 'longitude', 'depth_km',
       'alert', 'tsunami', 'sig'],
      dtype='str')
/n Количество дубликатов по time: 0

 normalized данные сохранены


,id,time,mag,place,latitude,longitude,depth_km,alert,tsunami,sig
0,us6000sa20,2026-02-18 22:40:28.752,4.6,"80 km E of Tomioka, Japan",37.3977,141.9228,43.546,None,False,326
1,us6000sa5d,2026-02-19 08:24:30.319,4.4,"11 km SW of Katsuura, Japan",35.0735,140.2253,10.000,None,False,298
2,us6000sadg,2026-02-20 10:39:18.090,4.5,"129 km ESE of Iwaki, Japan",36.6758,142.2563,10.000,None,False,312
3,us6000sae0,2026-02-20 14:16:30.267,5.3,"71 km ENE of Mutsu, Japan",41.5427,141.9999,54.152,None,False,432
4,us6000sai0,2026-02-20 18:59:32.395,4.2,"88 km SSE of Onagawa Chō, Japan",37.7788,142.0091,51.363,None,False,271
5,us6000samv,2026-02-21 17:09:06.318,4.3,"53 km NE of Misawa, Japan",40.9776,141.8946,65.940,None,False,284
6,us6000saxx,2026-02-23 02:59:33.402,4.6,"86 km ENE of Noda, Japan",40.4445,142.7401,52.286,None,False,326
7,us7000s2l1,2026-02-23 03:50:10.969,4.2,"94 km ENE of Noda, Japan",40.4307,142.8475,54.739,None,False,271
8,us7000rzvn,2026-02-23 22:34:21.130,4.5,"59 km SE of Shimoda, Japan",34.3291,139.4449,139.765,None,False,312
9,us7000rzvv,2026-02-23 23:03:41.033,4.8,"7 km E of Nanao, Japan",37.0384,137.0444,259.384,None,False,354


In [7]:
clean_df['date'] = clean_df['time'].dt.date

# 1. Количество событий по дням (cnt_events)
daily_events = clean_df.groupby('date').agg(
    cnt_events=('id', 'count')
).reset_index()
print("\n1. Количество событий по дням:")
print(daily_events.head())

# 2. Статистика по магнитудам (максимальная, средняя, минимальная) по дням
daily_mag_stats = clean_df.groupby('date').agg(
    max_mag_day=('mag', 'max'),
    avg_mag=('mag', 'mean'),
    min_mag_day=('mag', 'min')
).reset_index()
print("\n2. Статистика по магнитудам по дням:")
print(daily_mag_stats.head())

# 3. Топ-10 событий по магнитуде с местом и глубиной
top10 = clean_df.nlargest(10, 'mag')[['mag', 'place', 'depth_km']]
print("\n3. Топ-10 событий по магнитуде:")
print(top10)

# 4. Доля глубоких событий depth_km > 70
deep_ratio = (clean_df['depth_km'] > 70).mean() * 100
print(f"\n4. Доля глубоких событий (depth_km > 70): {deep_ratio:.2f}%")

# ========== ВЫВОД ==========
print("\n=== Информация о данных ===")
print("head():")
display(clean_df.head())
print(f"\nshape: {clean_df.shape}")
print(f"\ndtypes:\n{clean_df.dtypes}")
print(f"\nТри KPI:")
print(f"- Всего событий: {len(clean_df)}")
print(f"- Максимальная магнитуда: {clean_df['mag'].max():.2f}")
print(f"- Средняя магнитуда: {clean_df['mag'].mean():.2f}")
print(f"- Минимальная магнитуда: {clean_df['mag'].min():.2f}")
print(f"- Доля глубоких событий: {deep_ratio:.2f}%")

# ========== СОХРАНЕНИЕ ВИТРИНЫ ==========
# Создание витрины с ежедневной гранулярностью
mart = daily_events.merge(daily_mag_stats, on='date', how='left')
mart['deep_events_percent'] = deep_ratio
mart['total_events'] = len(clean_df)
mart['max_magnitude_overall'] = clean_df['mag'].max()
mart['region_id'] = 'JP_HON'
mart['region_name'] = 'Япония (Хонсю)'

# Сохранение витрины
mart.to_csv(f"../data/mart/variant_16/mart_daily_{file_name}.csv", index=False, encoding='utf-8')
print(f"\n✓ Витрина сохранена: ../data/mart/variant_16/mart_daily_{file_name}.csv")
print(f"  Количество дней: {len(mart)}")
print(f"\nСтруктура витрины:")
print(mart.head())


1. Количество событий по дням:
         date  cnt_events
0  2026-02-18           1
1  2026-02-19           1
2  2026-02-20           3
3  2026-02-21           1
4  2026-02-23           4

2. Статистика по магнитудам по дням:
         date  max_mag_day   avg_mag  min_mag_day
0  2026-02-18          4.6  4.600000          4.6
1  2026-02-19          4.4  4.400000          4.4
2  2026-02-20          5.3  4.666667          4.2
3  2026-02-21          4.3  4.300000          4.3
4  2026-02-23          4.8  4.525000          4.2

3. Топ-10 событий по магнитуде:
    mag                       place  depth_km
28  5.9    19 km NE of Otobe, Japan   144.000
24  5.7   110 km E of Yamada, Japan    16.000
18  5.5  21 km SW of Ibusuki, Japan   127.494
23  5.4   109 km E of Yamada, Japan    35.000
3   5.3   71 km ENE of Mutsu, Japan    54.152
13  5.2    96 km S of Sukumo, Japan    17.334
34  5.0   116 km E of Yamada, Japan    25.896
15  4.9  65 km E of Nichinan, Japan    10.000
26  4.9   Izu Islands, Japa

,id,time,mag,place,latitude,longitude,depth_km,alert,tsunami,sig,date
0,us6000sa20,2026-02-18 22:40:28.752,4.6,"80 km E of Tomioka, Japan",37.3977,141.9228,43.546,None,False,326,2026-02-18
1,us6000sa5d,2026-02-19 08:24:30.319,4.4,"11 km SW of Katsuura, Japan",35.0735,140.2253,10.000,None,False,298,2026-02-19
2,us6000sadg,2026-02-20 10:39:18.090,4.5,"129 km ESE of Iwaki, Japan",36.6758,142.2563,10.000,None,False,312,2026-02-20
3,us6000sae0,2026-02-20 14:16:30.267,5.3,"71 km ENE of Mutsu, Japan",41.5427,141.9999,54.152,None,False,432,2026-02-20
4,us6000sai0,2026-02-20 18:59:32.395,4.2,"88 km SSE of Onagawa Chō, Japan",37.7788,142.0091,51.363,None,False,271,2026-02-20



shape: (47, 11)

dtypes:
id                      str
time         datetime64[ms]
mag                 float64
place                   str
latitude            float64
longitude           float64
depth_km            float64
alert                   str
tsunami                bool
sig                   int64
date                 object
dtype: object

Три KPI:
- Всего событий: 47
- Максимальная магнитуда: 5.90
- Средняя магнитуда: 4.63
- Минимальная магнитуда: 4.00
- Доля глубоких событий: 23.40%

✓ Витрина сохранена: ../data/mart/variant_16/mart_daily_2026-03-20_11-46-40.csv
  Количество дней: 21

Структура витрины:
         date  cnt_events  max_mag_day   avg_mag  min_mag_day  \
0  2026-02-18           1          4.6  4.600000          4.6   
1  2026-02-19           1          4.4  4.400000          4.4   
2  2026-02-20           3          5.3  4.666667          4.2   
3  2026-02-21           1          4.3  4.300000          4.3   
4  2026-02-23           4          4.8  4.525000       